# Cortex AI for Indirect Tax

**Lab 2 of 2** · 60 min core + 34 min optional + 2 min cleanup · Intermediate

Read the invoice narrative, catch the miscoding, draft the working paper, and answer the partner in plain English — all inside Snowflake.

This lab depends on the objects you built in Lab 1. Run the verification cell first - if anything is missing, go back and finish Lab 1.

**Before you start**, you need the `SNOWFLAKE.CORTEX_USER` database role. Set the notebook role to `SYSADMIN` and the warehouse to `GSI_TAX_WH`.

> Full narrative, expected results and architecture notes: **`gsi_tax_workshop_lab2_cortex_ai.html`** — keep it open alongside this notebook.

---

> ⚠️ **Synthetic data**
>
> All data in this workshop is synthetically generated for training purposes. Entity names, ABNs, suppliers and transactions are fictitious. Tax treatments shown are simplified for illustration and are not tax advice.

---

## Step 1 — Verify your Lab 1 foundation

*~3 min* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-1)

Lab 2 adds one thing to what you already built: the ability to read. Everything else — the conformed ledger, the determination rules, the reconciliation, the exception register — is already on disk from Lab 1, and this lab does not rebuild any of it. Before you spend a single language model call, confirm the foundation is intact.

> ⚠️ **Prerequisites**
>
> **Lab 1 must be complete.** Every object this lab reads from was created there, and nothing in Lab 2 will run without it. You also need the `SNOWFLAKE.CORTEX_USER` database role, which is what grants access to the Cortex AI functions.

### One query, all four layers

The query below counts every object the lab depends on and puts the expected count beside the actual one, so you diagnose your own environment rather than comparing numbers against a screen at the front of the room.

In [ ]:
USE DATABASE GSI_TAX_WORKSHOP;
USE WAREHOUSE GSI_TAX_WH;

SELECT 1 AS SORT_KEY, 'RAW'      AS LAYER, 'RAW_DATA.ERP_SAP_GL'     AS OBJECT_NAME,
       COUNT(*) AS ACTUAL_ROWS, 1200000 AS EXPECTED_ROWS,
       IFF(COUNT(*) = 1200000, 'OK', 'CHECK') AS STATUS
FROM RAW_DATA.ERP_SAP_GL
UNION ALL
SELECT 2, 'RAW',     'RAW_DATA.ERP_ORACLE_GL',  COUNT(*),  800000,
       IFF(COUNT(*) =  800000, 'OK', 'CHECK') FROM RAW_DATA.ERP_ORACLE_GL
UNION ALL
SELECT 3, 'RAW',     'RAW_DATA.ERP_LEGACY_AP',  COUNT(*),  400000,
       IFF(COUNT(*) =  400000, 'OK', 'CHECK') FROM RAW_DATA.ERP_LEGACY_AP
UNION ALL
SELECT 4, 'RAW',     'RAW_DATA.DOC_TAX_INVOICE', COUNT(*),     300,
       IFF(COUNT(*) =     300, 'OK', 'CHECK') FROM RAW_DATA.DOC_TAX_INVOICE
UNION ALL
SELECT 5, 'CURATED', 'CURATED.GL_HARMONISED',   COUNT(*), 2400000,
       IFF(COUNT(*) = 2400000, 'OK', 'CHECK') FROM CURATED.GL_HARMONISED
UNION ALL
SELECT 6, 'TAX',     'TAX.V_GST_DETERMINED',    COUNT(*), 2400000,
       IFF(COUNT(*) = 2400000, 'OK', 'CHECK') FROM TAX.V_GST_DETERMINED
UNION ALL
SELECT 7, 'TAX',     'TAX.V_GST_VS_BAS',        COUNT(*),     532,
       IFF(COUNT(*) =     532, 'OK', 'CHECK') FROM TAX.V_GST_VS_BAS
UNION ALL
SELECT 8, 'TAX',     'TAX.V_GST_EXCEPTIONS',    COUNT(*),  258145,
       IFF(COUNT(*) =  258145, 'OK', 'CHECK') FROM TAX.V_GST_EXCEPTIONS
ORDER BY SORT_KEY;

Eight rows, and every `STATUS` should read `OK`.

| Layer | Object | Expected rows | What it is |
|---|---|---|---|
| RAW | `RAW_DATA.ERP_SAP_GL` | 1,200,000 | Source ledger, signed amounts, text dates. |
| RAW | `RAW_DATA.ERP_ORACLE_GL` | 800,000 | Source ledger, separate debit and credit columns. |
| RAW | `RAW_DATA.ERP_LEGACY_AP` | 400,000 | Source ledger, GST-inclusive amounts. |
| RAW | `RAW_DATA.DOC_TAX_INVOICE` | 300 | The free text this lab reads. Untouched in Lab 1. |
| CURATED | `CURATED.GL_HARMONISED` | 2,400,000 | The one governed ledger. The only Dynamic Table here. |
| TAX | `TAX.V_GST_DETERMINED` | 2,400,000 | Every line with its treatment, rate and rule lineage. |
| TAX | `TAX.V_GST_VS_BAS` | 532 | Entity by period reconciliation against lodged BAS. |
| TAX | `TAX.V_GST_EXCEPTIONS` | 253,394 | The work queue. 10.56% of the ledger. |

> ⚠️ **If any count is zero**
>
> A zero means the object exists but was never populated, or the schema was created and the load never ran. Re-run the Lab 1 Step 1 setup script in full and then re-run this query.

The 300-row table is the one to notice. It is the only object in the account holding free text, it has been sitting there since Lab 1 completely unused, and it is where the next four steps happen. Everything a human wrote about these transactions is in there, and so far nothing in your pipeline has been able to do anything with it.

> ✅ **Where you are**
>
> Foundation verified across four layers. The governed ledger, the determination rules and the exception register are all intact and queryable.

---

## Step 2 — Classify the supply from the narrative

*~15 min* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-2)

Supply category drives tax treatment. In Lab 1 it arrived through a join to `DIM_ACCOUNT_MAPPING`, which means it is really a property of the general ledger account somebody chose at data entry — a judgement made in seconds, by a person who is not in the tax team, against a chart of accounts they did not design. That is where miscoding is born, and no amount of rule logic downstream can detect it, because the rules faithfully apply the correct treatment to the wrong category.

- Read the invoice narrative that has been sitting unused since Lab 1.
- Have a language model classify the supply from that narrative in one function call.
- Compare what the AI read against what the ledger booked, and find the disagreements.
- Push the findings into the exception register the tax team already works from.

### Block 1 — the narrative, the booking and the AI verdict in one grid

Three columns: what the supplier wrote, what the ledger recorded, and what the model concluded. Where columns two and three disagree, you are looking at a candidate miscoding — and you are looking at it without having written a single rule about what a client dinner is.

In [ ]:
-- The LIMIT sits INSIDE the CTE so it bounds the rows fed to the model,
-- not just the rows displayed. This matters: AI cost scales with rows.
WITH doc_sample AS (
    SELECT DOC_ID, SOURCE_SYSTEM, SOURCE_DOC_REF,
           INVOICE_NARRATIVE, BOOKED_SUPPLY_CATEGORY
    FROM RAW_DATA.DOC_TAX_INVOICE
    LIMIT 10
)
SELECT
    DOC_ID,
    INVOICE_NARRATIVE,
    BOOKED_SUPPLY_CATEGORY,
    AI_CLASSIFY(
        INVOICE_NARRATIVE,
        ['taxable supply','GST-free supply','input taxed supply',
         'export','entertainment','capital acquisition']
    ):labels[0]::VARCHAR AS AI_SUPPLY_CATEGORY
FROM doc_sample;

Read across the rows. *"Monthly external audit and assurance fees for the year ended 30 June"* returns `taxable supply`. *"Forklift purchase, capitalised to plant and equipment"* returns `capital acquisition`. *"Sea freight Melbourne to Auckland, export consignment, FOB terms"* returns `export`. No prompt engineering, no training data, no model to host — one function, a list of your six categories, and the answer arrives as structured JSON you can index into.

> 💡 **The syntax, once, because you will reuse it**
>
> `AI_CLASSIFY` returns an object, not a string. `:labels[0]` takes the first label from the returned array and `::VARCHAR` casts it to text so it can be compared, grouped and joined like any other column.

> ⚠️ **Why this table has 300 rows and not 2,400,000**
>
> `DOC_TAX_INVOICE` holds **300 documents**, not the 2.4 million lines in the ledger, and that is a deliberate design decision rather than a shortcut for the workshop. Language model cost scales linearly with rows: running a classifier across the full ledger would cost thousands of times what this step costs and tell you almost nothing extra.

### Block 2 — make it a view so the rest of the lab can join to it

The classification is only useful if it is an object. The view below applies it across all 300 documents and adds the one column that turns an observation into a finding: a flag comparing what the AI read to what the ledger booked. The `CASE` expression exists because the two vocabularies differ — the ledger uses eleven account-level categories, the classifier uses the six tax-relevant ones — so the booked category is mapped onto the classifier's vocabulary before the comparison is made.

In [ ]:
CREATE OR REPLACE VIEW AI_INSIGHTS.V_AI_SUPPLY_CLASSIFIED AS
WITH classified AS (
    SELECT
        d.DOC_ID,
        d.SOURCE_SYSTEM,
        d.SOURCE_DOC_REF,
        d.SUPPLIER_ID,
        d.INVOICE_NARRATIVE,
        d.BOOKED_SUPPLY_CATEGORY,
        AI_CLASSIFY(
            d.INVOICE_NARRATIVE,
            ['taxable supply','GST-free supply','input taxed supply',
             'export','entertainment','capital acquisition']
        ):labels[0]::VARCHAR AS AI_SUPPLY_CATEGORY
    FROM RAW_DATA.DOC_TAX_INVOICE d
)
SELECT
    c.*,
    -- Map the ledger's account-level category onto the classifier's
    -- tax-relevant vocabulary so the two can be compared at all.
    CASE c.BOOKED_SUPPLY_CATEGORY
        WHEN 'ENTERTAINMENT'     THEN 'entertainment'
        WHEN 'CAPITAL_EQUIPMENT' THEN 'capital acquisition'
        WHEN 'EXPORT_SALES'      THEN 'export'
        WHEN 'RESIDENTIAL_RENT'  THEN 'input taxed supply'
        WHEN 'FINANCIAL_SUPPLY'  THEN 'input taxed supply'
        WHEN 'FOOD_BASIC'        THEN 'GST-free supply'
        WHEN 'MEDICAL'           THEN 'GST-free supply'
        WHEN 'EDUCATION'         THEN 'GST-free supply'
        ELSE 'taxable supply'
    END                                                    AS BOOKED_EQUIVALENT,
    (AI_SUPPLY_CATEGORY IS NOT NULL
     AND AI_SUPPLY_CATEGORY <> CASE c.BOOKED_SUPPLY_CATEGORY
        WHEN 'ENTERTAINMENT'     THEN 'entertainment'
        WHEN 'CAPITAL_EQUIPMENT' THEN 'capital acquisition'
        WHEN 'EXPORT_SALES'      THEN 'export'
        WHEN 'RESIDENTIAL_RENT'  THEN 'input taxed supply'
        WHEN 'FINANCIAL_SUPPLY'  THEN 'input taxed supply'
        WHEN 'FOOD_BASIC'        THEN 'GST-free supply'
        WHEN 'MEDICAL'           THEN 'GST-free supply'
        WHEN 'EDUCATION'         THEN 'GST-free supply'
        ELSE 'taxable supply'
     END)                                                  AS MISCODING_FLAG
FROM classified c;

Note the `IS NOT NULL` guard inside `MISCODING_FLAG`. If the model ever returns nothing for a narrative, a bare `<>` comparison would yield `NULL` rather than `TRUE`, and a `NULL` flag quietly drops out of every `WHERE MISCODING_FLAG` filter downstream. An unclassified document is not a clean document, and you do not want it disappearing from the count.

### Block 3 — the miscoding pattern

Individual disagreements are interesting. The pattern across them is what a tax practice acts on, because it points at a process rather than a transaction.

In [ ]:
SELECT
    BOOKED_SUPPLY_CATEGORY,
    AI_SUPPLY_CATEGORY,
    COUNT(*)                                                     AS DOC_COUNT,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)           AS PCT_OF_SAMPLE
FROM AI_INSIGHTS.V_AI_SUPPLY_CLASSIFIED
WHERE MISCODING_FLAG
GROUP BY BOOKED_SUPPLY_CATEGORY, AI_SUPPLY_CATEGORY
ORDER BY DOC_COUNT DESC;

One row of this result matters more than the others: **entertainment booked as professional fees**. The rest are largely vocabulary drift — a training course booked to general services, a canteen delivery booked to consumables — and they cost nothing, because the treatment happens to land in the same place either way.

Entertainment is different. GST is genuinely charged on a corporate box or a client dinner, so the invoice looks unremarkable and the amount reconciles. The credit, however, is **denied** where the underlying expense is non-deductible. Book that same dinner to professional fees and the input tax credit is claimed automatically, correctly according to the rules, on a supply where it was never available. Nothing in the ledger is inconsistent. Nothing in the reconciliation breaks. The only evidence that anything is wrong is the sentence the supplier wrote on the invoice.

> 💡 **Why this is the finding a reviewer wants**
>
> An over-claimed credit on a denied supply is a real exposure with interest and penalties attached, and it is invisible to every control that operates on numbers. It is only visible to a control that reads.

### Block 4 — close the loop into the existing work queue

A finding that lives in its own table is a finding nobody actions. The join below lands the AI results in `TAX.V_GST_EXCEPTIONS` — the same register the tax team has been working from since Lab 1 — on `(SOURCE_SYSTEM, SOURCE_DOC_REF)`.

In [ ]:
SELECT
    c.DOC_ID,
    c.SOURCE_SYSTEM,
    c.SOURCE_DOC_REF,
    c.BOOKED_SUPPLY_CATEGORY,
    c.AI_SUPPLY_CATEGORY,
    e.ENTITY_NAME,
    e.SUPPLIER_NAME,
    e.TAX_TREATMENT,
    e.APPLIED_RULE_ID,
    e.EXCEPTION_CODES,
    e.NET_AMOUNT,
    e.BOOKED_TAX_AMOUNT,
    e.TAX_VARIANCE,
    e.RISK_SCORE
FROM AI_INSIGHTS.V_AI_SUPPLY_CLASSIFIED c
JOIN TAX.V_GST_EXCEPTIONS e
  ON e.SOURCE_SYSTEM  = c.SOURCE_SYSTEM
 AND e.SOURCE_DOC_REF = c.SOURCE_DOC_REF
WHERE c.MISCODING_FLAG
ORDER BY e.RISK_SCORE DESC
LIMIT 20;

Every AI-flagged document now arrives carrying the entity, the supplier, the treatment that was applied, **the rule that applied it** and the existing exception codes on the same line. A reviewer opening this result has the AI's opinion and the deterministic audit trail in a single row, and can disagree with the former without touching the latter.

> ✅ **The teaching point — AI did not replace the rules**
>
> Read the determination logic again: it is unchanged. No rule was rewritten, no rate was moved, no treatment was decided by a model.

---

## Step 3 — Extract structured fields from free text

*~12 min* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-3)

Classification answers "what kind of supply is this?". It does not answer "does this document actually support the credit we claimed?" That question is answered by specific facts printed on the invoice — the supplier's ABN and the GST charged — and those facts are sitting in a free-text remarks field that no report has ever read.

- Pull two named values out of unstructured text with `EXTRACT_ANSWER`.
- Reconcile the extracted ABN against the supplier master.
- Quantify the input tax credit at risk where no ABN exists anywhere.

> 💡 **The Australian rule this step tests**
>
> For a taxable supply of more than **$82.50** including GST, a recipient claiming an input tax credit must hold a valid tax invoice, and that invoice must show the **supplier's ABN**. No ABN means no valid tax invoice, and no valid tax invoice means **no input tax credit** — regardless of whether the expense was genuine, the money was paid …

### Block 1 — two questions, asked of the same text

`EXTRACT_ANSWER` takes a document and a question in plain English. It is not a regular expression: it does not need to know that an ABN is eleven digits in a 2-3-3-3 grouping, and it does not break when the surrounding sentence is phrased differently.

In [ ]:
WITH doc_sample AS (
    SELECT DOC_ID, SOURCE_SYSTEM, SOURCE_DOC_REF, SUPPLIER_REMARKS
    FROM RAW_DATA.DOC_TAX_INVOICE
    LIMIT 10
)
SELECT
    DOC_ID,
    SUPPLIER_REMARKS,
    SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
        SUPPLIER_REMARKS,
        'What is the supplier ABN?'
    )[0]:answer::VARCHAR AS EXTRACTED_ABN,
    SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
        SUPPLIER_REMARKS,
        'What GST amount is stated on this invoice?'
    )[0]:answer::VARCHAR AS EXTRACTED_GST_TEXT
FROM doc_sample;

The ABN column fills in cleanly wherever the remarks contain one. The GST column is the more instructive of the two, and it is worth pausing on rather than skipping past.

Roughly two thirds of these documents say *"GST included in the total shown"* — which is a perfectly valid statement under Australian GST law and is **not a number**. So the extraction returns a fragment of that phrase, or the ABN digits it found nearby, rather than an amount. That is the model behaving correctly on text that does not contain the answer, and the reviewer's conclusion is the right one: where the invoice states GST as included rather than stated, you fall back to the GST fraction — one eleventh of the GST-inclusive total — and you record that you did. An extraction that returns nothing usable is a finding about the document, not a failure of the tool.

> ⚠️ **Do not treat extracted text as trusted numeric input**
>
> `EXTRACT_ANSWER` returns a string, and a string that looks like a number is not a number. Wrap every numeric conversion in `TRY_TO_NUMBER` so an unparseable answer arrives as `NULL` and can be counted, exactly as `TRY_TO_DATE` handled the unparseable legacy dates in Lab 1.

### Block 2 — the extraction as a queryable object

The view below extracts both fields across all 300 documents, normalises the ABN for comparison by stripping every non-digit character from both sides, and attaches the ledger's own view of the same document: the supplier master ABN and the tax actually booked. Because one document reference can carry several ledger lines, the booked tax is aggregated per document **before** the join, so no document is counted twice.

In [ ]:
CREATE OR REPLACE VIEW AI_INSIGHTS.V_AI_DOC_EXTRACTED AS
WITH extracted AS (
    SELECT
        d.DOC_ID,
        d.SOURCE_SYSTEM,
        d.SOURCE_DOC_REF,
        d.SUPPLIER_ID,
        d.SUPPLIER_REMARKS,
        SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
            d.SUPPLIER_REMARKS, 'What is the supplier ABN?'
        )[0]:answer::VARCHAR AS EXTRACTED_ABN_RAW,
        SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
            d.SUPPLIER_REMARKS, 'What GST amount is stated on this invoice?'
        )[0]:answer::VARCHAR AS EXTRACTED_GST_RAW
    FROM RAW_DATA.DOC_TAX_INVOICE d
),
doc_tax AS (
    -- Aggregate to one row per document BEFORE joining, so a multi-line
    -- document cannot fan out and double-count the tax.
    SELECT SOURCE_SYSTEM, SOURCE_DOC_REF,
           SUM(NET_AMOUNT)        AS DOC_NET_AMOUNT,
           SUM(BOOKED_TAX_AMOUNT) AS BOOKED_TAX_AMOUNT,
           SUM(GROSS_AMOUNT)      AS DOC_GROSS_AMOUNT,
           MAX(DOC_TYPE)          AS DOC_TYPE,
           MAX(ENTITY_NAME)       AS ENTITY_NAME,
           MAX(TAX_PERIOD)        AS TAX_PERIOD,
           BOOLOR_AGG(RECOVERABLE) AS RECOVERABLE
    FROM TAX.V_GST_DETERMINED
    GROUP BY SOURCE_SYSTEM, SOURCE_DOC_REF
)
SELECT
    x.DOC_ID,
    x.SOURCE_SYSTEM,
    x.SOURCE_DOC_REF,
    x.SUPPLIER_ID,
    s.SUPPLIER_NAME,
    t.ENTITY_NAME,
    t.TAX_PERIOD,
    t.DOC_TYPE,
    t.RECOVERABLE,
    x.SUPPLIER_REMARKS,
    x.EXTRACTED_ABN_RAW,
    x.EXTRACTED_GST_RAW,
    -- Strip everything that is not a digit from BOTH sides so the
    -- comparison is not defeated by spacing conventions.
    REGEXP_REPLACE(COALESCE(x.EXTRACTED_ABN_RAW, ''), '[^0-9]', '') AS EXTRACTED_ABN_DIGITS,
    REGEXP_REPLACE(COALESCE(s.SUPPLIER_ABN,      ''), '[^0-9]', '') AS MASTER_ABN_DIGITS,
    TRY_TO_NUMBER(REGEXP_REPLACE(COALESCE(x.EXTRACTED_GST_RAW, ''), '[^0-9.]', '')) AS EXTRACTED_GST_AMOUNT,
    t.DOC_NET_AMOUNT,
    t.DOC_GROSS_AMOUNT,
    t.BOOKED_TAX_AMOUNT,
    -- The fallback every reviewer uses when the invoice states GST as
    -- included rather than stated: one eleventh of the GST-inclusive total.
    ROUND(COALESCE(t.DOC_GROSS_AMOUNT, 0) / 11, 2)                  AS GST_FRACTION_AMOUNT,
    (LENGTH(REGEXP_REPLACE(COALESCE(x.EXTRACTED_ABN_RAW, ''), '[^0-9]', '')) = 11) AS HAS_ABN_ON_INVOICE,
    (s.SUPPLIER_ABN IS NULL)                                        AS MASTER_ABN_MISSING
FROM extracted x
LEFT JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = x.SUPPLIER_ID
LEFT JOIN doc_tax t
       ON t.SOURCE_SYSTEM  = x.SOURCE_SYSTEM
      AND t.SOURCE_DOC_REF = x.SOURCE_DOC_REF;

Two things in that statement are worth carrying into your own work. The ABN comparison strips non-digits from both the extracted value and the master value, because `75 370 698 317` and `75370698317` are the same ABN and a naive string comparison says they are not. And `HAS_ABN_ON_INVOICE` tests for exactly eleven digits rather than for a non-empty string, so a partial extraction does not register as a valid ABN.

### Block 3 — one query, three findings

Three separate questions, three separate reports and three separate meetings is how this work usually gets done. It fits in one result set.

In [ ]:
SELECT
    COUNT(*)                                                     AS DOCS_REVIEWED,

    -- Finding 1: the invoice shows an ABN, and it is not the supplier's.
    COUNT_IF(HAS_ABN_ON_INVOICE
             AND MASTER_ABN_DIGITS <> ''
             AND EXTRACTED_ABN_DIGITS <> MASTER_ABN_DIGITS)      AS ABN_MISMATCH_DOCS,

    -- Finding 2: the invoice states a GST amount that does not agree with
    -- the tax the ledger booked, beyond a one dollar rounding tolerance.
    COUNT_IF(EXTRACTED_GST_AMOUNT IS NOT NULL
             AND ABS(EXTRACTED_GST_AMOUNT - COALESCE(BOOKED_TAX_AMOUNT, 0)) > 1.00)
                                                                 AS GST_MISMATCH_DOCS,

    -- Finding 3: no ABN on the invoice AND none in the supplier master.
    -- On a recoverable AP invoice over $82.50 there is no valid tax
    -- invoice, so there is no input tax credit.
    COUNT_IF(NOT HAS_ABN_ON_INVOICE AND MASTER_ABN_MISSING)       AS NO_ABN_ANYWHERE_DOCS,
    ROUND(SUM(CASE WHEN NOT HAS_ABN_ON_INVOICE
                    AND MASTER_ABN_MISSING
                    AND RECOVERABLE
                    AND DOC_TYPE = 'AP_INVOICE'
                    AND COALESCE(DOC_GROSS_AMOUNT, 0) > 82.50
                   THEN COALESCE(BOOKED_TAX_AMOUNT, 0) ELSE 0 END), 2)
                                                                 AS ITC_AT_RISK_AUD
FROM AI_INSIGHTS.V_AI_DOC_EXTRACTED;

One row, and every column on it is a conversation with a client. An ABN on the invoice that does not match the supplier master is either a master data error or a supplier you are not actually transacting with. A stated GST amount that disagrees with the booked tax is a posting error on a document you hold the evidence for. And `ITC_AT_RISK_AUD` is the number that ends the discussion: credits claimed where no valid tax invoice can be produced, in dollars, from the primary evidence rather than from an assertion.

> ⚠️ **Read the $82.50 condition carefully**
>
> The exposure calculation is deliberately narrow: recoverable AP invoices only, over the $82.50 threshold only, and only where the ABN is absent from **both** the invoice and the supplier master. Each of those conditions removes a class of false positive — an AR invoice you issued, a low-value acquisition below the threshold, a supplier whose ABN you already hold on …

> ✅ **The teaching point — the invoice is the primary evidence**
>
> In an audit, the ledger is a claim and the invoice is the evidence for it. Until this step your pipeline could interrogate the claim in 2.4 million rows and could not read a single piece of evidence, so every ABN and GST-amount test was a manual sample of a filing cabinet.

---

## Step 4 — Draft the working papers

*~14 min* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-4)

The exception register found 253,394 lines in Lab 1. Finding them was never the hard part. The hard part is that somebody has to write, for each one, a paragraph explaining what happened, which rule applied, why it matters and what the reviewer should do — and that paragraph is the bottleneck between a query result and a deliverable a client will pay for.

- See what a lazy prompt produces, next to what a specified prompt produces.
- Ground the generated text in the rule that actually fired, so it cannot invent one.
- Build the rationale as a bounded view and read the working papers back.

### Block 1 — the same exceptions, two prompts, one grid

Both columns below are generated by the same model, over the same five exceptions, in the same query. The only variable is the prompt. Read the two side by side before you read anything else in this step.

In [ ]:
WITH top_exceptions AS (
    -- LIMIT inside the CTE: five exceptions reach the model, not 253,394.
    SELECT e.GL_LINE_ID, e.ENTITY_NAME, e.SUPPLIER_NAME, e.TAX_PERIOD,
           e.SUPPLY_CATEGORY, e.TAX_TREATMENT, e.APPLIED_RULE_ID,
           e.EXCEPTION_CODES, e.EXCEPTION_REASONS, e.MAX_SEVERITY,
           e.NET_AMOUNT, e.BOOKED_TAX_AMOUNT, e.EXPECTED_TAX_AMOUNT,
           e.TAX_VARIANCE, e.RISK_SCORE,
           r.RULE_DESCRIPTION AS APPLIED_RULE_DESCRIPTION
    FROM TAX.V_GST_EXCEPTIONS e
    LEFT JOIN TAX.TAX_RULE_DETERMINATION r ON r.RULE_ID = e.APPLIED_RULE_ID
    ORDER BY e.RISK_SCORE DESC
    LIMIT 5
)
SELECT
    GL_LINE_ID,
    EXCEPTION_CODES,
    ROUND(TAX_VARIANCE, 2) AS TAX_VARIANCE,

    -- BAD PROMPT: no role, no audience, no length, no grounding.
    AI_COMPLETE('claude-sonnet-4-6',
        'Explain this tax problem: ' || COALESCE(EXCEPTION_REASONS, 'none')
    )                                                        AS BAD_PROMPT_OUTPUT,

    -- GOOD PROMPT: role, audience, length, required elements, and the
    -- rule that actually fired passed in as grounding.
    AI_COMPLETE('claude-sonnet-4-6',
        'You are an Australian indirect tax specialist preparing a working '  ||
        'paper for a review partner. Write exactly three sentences in '       ||
        'British English, no headings and no bullet points. Sentence one: '   ||
        'state what the exception is. Sentence two: state why it matters '    ||
        'for GST, referring to the determination rule provided. Sentence '    ||
        'three: state the specific next step for the reviewer. Do not '       ||
        'speculate beyond the facts given.'                                   ||
        ' FACTS -- Entity: '            || COALESCE(ENTITY_NAME, 'unknown')    ||
        '; Supplier: '                  || COALESCE(SUPPLIER_NAME, 'unknown')  ||
        '; Tax period: '                || COALESCE(TAX_PERIOD::VARCHAR, 'unknown') ||
        '; Supply category: '           || COALESCE(SUPPLY_CATEGORY, 'unmapped')     ||
        '; Determined treatment: '      || COALESCE(TAX_TREATMENT, 'undetermined')   ||
        '; Determination rule applied: '|| COALESCE(APPLIED_RULE_DESCRIPTION, 'none')||
        '; Exception codes: '           || COALESCE(EXCEPTION_CODES, 'none')         ||
        '; Exception reasons: '         || COALESCE(EXCEPTION_REASONS, 'none')       ||
        '; Net amount AUD: '            || COALESCE(NET_AMOUNT::VARCHAR, '0')        ||
        '; Booked tax AUD: '            || COALESCE(BOOKED_TAX_AMOUNT::VARCHAR, '0') ||
        '; Expected tax AUD: '          || COALESCE(EXPECTED_TAX_AMOUNT::VARCHAR, '0')||
        '; Tax variance AUD: '          || COALESCE(TAX_VARIANCE::VARCHAR, '0')       ||
        '; Severity: '                  || COALESCE(MAX_SEVERITY, 'unknown')
    )                                                        AS GOOD_PROMPT_OUTPUT
FROM top_exceptions;

The bad output is generic, unusable and confident — a paragraph about GST in general, with no entity, no amount and no rule, that a reviewer must discard and rewrite. The good output names the entity, quotes the variance, cites the determination rule that fired and ends with an instruction. Same model. Same data. Same cost, near enough. The difference is entirely in what you asked for.

> 💡 **Four things the good prompt specifies, and the bad one does not**
>
> **Role** — an Australian indirect tax specialist, so the vocabulary and the framing are professional rather than explanatory. **Audience** — a review partner, not a taxpayer, so it does not define what GST is.

> ⚠️ **Grounding is not a nicety — it is the control**
>
> Passing `APPLIED_RULE_DESCRIPTION` into the prompt is what stops the model inventing a rule that sounds plausible. It is not asked to determine the treatment; the determination already happened, deterministically, in Lab 1.

### Block 2 — the rationale as a bounded view

The view below uses only the good prompt, carries the rule lineage through to the output, and caps the population at 25 documents inside the CTE. The cap is inside the CTE deliberately: put `LIMIT 25` on the outer query instead and Snowflake may still invoke the model on everything it scanned before discarding rows.

In [ ]:
CREATE OR REPLACE VIEW AI_INSIGHTS.V_AI_EXCEPTION_RATIONALE AS
WITH ranked AS (
    -- The bound lives HERE. 25 rows reach the model, no matter how the
    -- view is queried downstream.
    SELECT e.GL_LINE_ID, e.ENTITY_ID, e.ENTITY_NAME, e.TAX_PERIOD,
           e.SUPPLIER_ID, e.SUPPLIER_NAME, e.SOURCE_SYSTEM, e.SOURCE_DOC_REF,
           e.DOC_TYPE, e.SUPPLY_CATEGORY, e.TAX_TREATMENT, e.APPLIED_RULE_ID,
           e.EXCEPTION_CODES, e.EXCEPTION_REASONS, e.EXCEPTION_COUNT,
           e.MAX_SEVERITY, e.NET_AMOUNT, e.BOOKED_TAX_AMOUNT,
           e.EXPECTED_TAX_AMOUNT, e.TAX_VARIANCE, e.RISK_SCORE,
           r.RULE_DESCRIPTION AS APPLIED_RULE_DESCRIPTION
    FROM TAX.V_GST_EXCEPTIONS e
    LEFT JOIN TAX.TAX_RULE_DETERMINATION r ON r.RULE_ID = e.APPLIED_RULE_ID
    ORDER BY e.RISK_SCORE DESC
    LIMIT 25
)
SELECT
    k.*,
    AI_COMPLETE('claude-sonnet-4-6',
        'You are an Australian indirect tax specialist preparing a working '  ||
        'paper for a review partner. Write exactly three sentences in '       ||
        'British English, no headings and no bullet points. Sentence one: '   ||
        'state what the exception is. Sentence two: state why it matters '    ||
        'for GST, referring to the determination rule provided. Sentence '    ||
        'three: state the specific next step for the reviewer. Do not '       ||
        'speculate beyond the facts given.'                                   ||
        ' FACTS -- Entity: '            || COALESCE(k.ENTITY_NAME, 'unknown')  ||
        '; Supplier: '                  || COALESCE(k.SUPPLIER_NAME, 'unknown')||
        '; Tax period: '                || COALESCE(k.TAX_PERIOD::VARCHAR, 'unknown') ||
        '; Source: '                    || COALESCE(k.SOURCE_SYSTEM, 'unknown')       ||
        ' document '                    || COALESCE(k.SOURCE_DOC_REF, 'unknown')      ||
        '; Document type: '             || COALESCE(k.DOC_TYPE, 'unknown')            ||
        '; Supply category: '           || COALESCE(k.SUPPLY_CATEGORY, 'unmapped')     ||
        '; Determined treatment: '      || COALESCE(k.TAX_TREATMENT, 'undetermined')   ||
        '; Determination rule applied: '|| COALESCE(k.APPLIED_RULE_DESCRIPTION, 'none')||
        '; Exception codes: '           || COALESCE(k.EXCEPTION_CODES, 'none')         ||
        '; Exception reasons: '         || COALESCE(k.EXCEPTION_REASONS, 'none')       ||
        '; Net amount AUD: '            || COALESCE(k.NET_AMOUNT::VARCHAR, '0')        ||
        '; Booked tax AUD: '            || COALESCE(k.BOOKED_TAX_AMOUNT::VARCHAR, '0') ||
        '; Expected tax AUD: '          || COALESCE(k.EXPECTED_TAX_AMOUNT::VARCHAR, '0')||
        '; Tax variance AUD: '          || COALESCE(k.TAX_VARIANCE::VARCHAR, '0')       ||
        '; Severity: '                  || COALESCE(k.MAX_SEVERITY, 'unknown')
    )                                                     AS WORKING_PAPER_NARRATIVE
FROM ranked k;

### Block 3 — read the working papers

In [ ]:
SELECT
    ENTITY_NAME,
    TAX_PERIOD,
    SUPPLIER_NAME,
    EXCEPTION_CODES,
    MAX_SEVERITY,
    APPLIED_RULE_ID,
    APPLIED_RULE_DESCRIPTION,
    ROUND(TAX_VARIANCE, 2) AS TAX_VARIANCE,
    ROUND(RISK_SCORE, 1)   AS RISK_SCORE,
    WORKING_PAPER_NARRATIVE
FROM AI_INSIGHTS.V_AI_EXCEPTION_RATIONALE
ORDER BY RISK_SCORE DESC;

Widen the `WORKING_PAPER_NARRATIVE` column in Snowsight and read three or four of them properly. Each is three sentences, in the same shape, naming the entity, the amount, the rule and the action. That is a first draft of a working paper produced from the exception register, and the reviewer's job has changed from writing to checking.

> ✅ **The teaching point — every sentence traces back to a rule row**
>
> `APPLIED_RULE_ID` and `APPLIED_RULE_DESCRIPTION` sit on the same row as the generated text, which means no paragraph in this output is unattributable. A partner who does not believe a sentence can look up rule 16 in `TAX.TAX_RULE_DETERMINATION`, read the row a human wrote, and see the validity dates that made it apply to this transaction.

> ⚠️ **This is the most expensive operation in the workshop**
>
> Generation costs materially more per row than classification or extraction, and it is the one place where an unbounded query becomes an unwelcome invoice. Three controls make it safe: `LIMIT` inside the CTE, ranking by `RISK_SCORE` so the spend goes to the exceptions that matter, and — in production — **materialising the output** so the model runs once rather than …

---

## Step 5 — Ask in plain English

*~16 min* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-5)

Everything so far has required somebody who can write SQL. The tax team cannot, and that is the actual constraint on how much of this work gets done — not compute, not data quality, not model availability. This step removes the SQL requirement and lets a tax specialist ask the question in the words they already use.

- Flatten the governed ledger, the exceptions and the BAS variance into one wide view.
- Create a semantic view — the business vocabulary the agent reasons over.
- Create the agent in Snowsight and attach the semantic view as an Analyst tool.
- Ask eight questions in plain English, and one of them closes the workshop.

### Block 1 — one wide view for the semantic layer

Cortex Analyst reasons best over a single wide, flat object: one row per ledger line, every dimension and every measure present as a column, no joins left for the model to work out. The view below assembles that from the three objects Lab 1 produced — determination, exceptions and reconciliation — joined once, here, so the agent never has to.

In [ ]:
CREATE OR REPLACE VIEW AGENTS.V_GST_INTELLIGENCE AS
SELECT
    -- keys and dimensions
    d.GL_LINE_ID,
    d.SOURCE_SYSTEM,
    d.SOURCE_DOC_REF,
    d.ENTITY_ID,
    d.ENTITY_NAME,
    d.JURISDICTION,
    d.DOC_DATE,
    d.TAX_PERIOD,
    d.DOC_TYPE,
    d.GROUP_ACCOUNT_CODE,
    d.GROUP_ACCOUNT_NAME,
    d.SUPPLY_CATEGORY,
    d.SUPPLIER_ID,
    d.SUPPLIER_NAME,
    d.SUPPLIER_ABN,
    d.SUPPLIER_COUNTRY,
    d.SUPPLIER_GST_REGISTERED,
    d.IS_CROSS_BORDER,
    d.TAX_TREATMENT,
    d.RECOVERABLE,
    d.APPLIED_RULE_ID,
    d.APPLIED_RULE_DESCRIPTION,
    d.APPLIED_RATE_PCT,
    d.IS_ACCOUNT_MAPPED,
    d.IS_DATE_PARSED,

    -- measures from the determination layer
    d.NET_AMOUNT,
    d.BOOKED_TAX_AMOUNT,
    d.EXPECTED_TAX_AMOUNT,
    d.TAX_VARIANCE,

    -- exception context, one row per line, so no fan-out is possible
    COALESCE(e.EXCEPTION_COUNT, 0)                       AS EXCEPTION_COUNT,
    (e.GL_LINE_ID IS NOT NULL)                           AS HAS_EXCEPTION,
    e.EXCEPTION_CODES,
    e.EXCEPTION_REASONS,
    COALESCE(e.MAX_SEVERITY, 'NONE')                     AS MAX_SEVERITY,
    COALESCE(e.RISK_SCORE, 0)                            AS RISK_SCORE,

    -- BAS reconciliation context, repeated per line within its entity/period
    b.COMPUTED_NET_GST,
    b.LODGED_NET_GST,
    b.VARIANCE_NET                                       AS BAS_VARIANCE_NET,
    b.VARIANCE_PCT                                       AS BAS_VARIANCE_PCT,
    COALESCE(b.RECON_STATUS, 'NOT_RECONCILED')           AS RECON_STATUS
FROM TAX.V_GST_DETERMINED d
LEFT JOIN TAX.V_GST_EXCEPTIONS e
       ON e.GL_LINE_ID = d.GL_LINE_ID
LEFT JOIN TAX.V_GST_VS_BAS b
       ON b.ENTITY_ID  = d.ENTITY_ID
      AND b.TAX_PERIOD = d.TAX_PERIOD;

Both joins are `LEFT JOIN`s for the same reason as in Lab 1: a line without an exception must still appear, or the agent's answer to "total GST payable" would silently exclude every clean transaction. `EXCEPTION_COUNT` and `RISK_SCORE` are wrapped in `COALESCE` so a clean line reads zero rather than `NULL`, which is what makes `SUM(EXCEPTION_COUNT)` answer the question a tax person actually asked.

> ⚠️ **The BAS columns repeat, and that is intentional**
>
> `V_GST_VS_BAS` is one row per entity per period, so joining it to a line-level view repeats the same reconciliation figures across every line in that period. Aggregating `LODGED_NET_GST` with `SUM` would therefore multiply it by the line count.

### The primary path — let Snowsight build the semantic view

Do this in the UI. It is faster than writing DDL, it produces a better result because it inspects the data before suggesting anything, and in a room it is considerably more convincing than reading a `CREATE` statement aloud.

1. In Snowsight, open **AI & ML**, then **Cortex Analyst**, then **Create new**.
2. Select `GSI_TAX_WORKSHOP.AGENTS.V_GST_INTELLIGENCE` as the source and let **Autopilot** run.
3. Review what it proposes. It will infer dimensions from the low-cardinality text and date columns, and metrics from the numeric ones, and it will be broadly right.
4. Correct the two things it cannot know. Set the BAS measures to **`MAX`** rather than `SUM`, for the reason in the callout above. And confirm that `EXPECTED_TAX_AMOUNT` and `BOOKED_TAX_AMOUNT` are described distinctly — one is what the rules say, the other is what the ERP posted, and an agent that conflates them will answer every variance question wrongly.
5. Then spend your remaining time on **synonyms**.

> 💡 **The synonyms are the product**
>
> A tax specialist does not type `LABEL_1B_GST_CREDITS`. They type *"input tax credits"*, or *"ITCs"*, or *"GST on purchases"*, and if the semantic view has never heard of any of those phrases the agent will guess or refuse.

### Block 2 — the DDL fallback

If Autopilot is unavailable in your account, or you want the semantic view in version control alongside the rest of the pipeline, this is the same object written by hand. Note that the synonyms and comments are not decoration: they are the entire interface between the tax vocabulary and the data.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW AGENTS.SV_GST_COMPLIANCE
    TABLES (
        gst AS AGENTS.V_GST_INTELLIGENCE
            PRIMARY KEY (GL_LINE_ID)
            WITH SYNONYMS = ('GST ledger','tax ledger','transactions','GST lines')
            COMMENT = 'One row per general ledger line with its determined Australian GST treatment, exception status and BAS reconciliation context.'
    )
    DIMENSIONS (
        gst.ENTITY_NAME AS ENTITY_NAME
            WITH SYNONYMS = ('entity','company','legal entity','business','subsidiary')
            COMMENT = 'Australian group entity that recorded the transaction.',
        gst.TAX_PERIOD AS TAX_PERIOD
            WITH SYNONYMS = ('period','month','tax period','BAS period','reporting period')
            COMMENT = 'Monthly GST period, stored as the first day of the month.',
        gst.DOC_DATE AS DOC_DATE
            WITH SYNONYMS = ('document date','transaction date','invoice date')
            COMMENT = 'Date of the underlying document. Determination rules are applied as at this date.',
        gst.SUPPLY_CATEGORY AS SUPPLY_CATEGORY
            WITH SYNONYMS = ('supply category','expense type','category','account category','spend type')
            COMMENT = 'Category of supply from the account mapping. NULL means the account is unmapped.',
        gst.TAX_TREATMENT AS TAX_TREATMENT
            WITH SYNONYMS = ('treatment','tax treatment','GST treatment','reverse charge','GST free','input taxed')
            COMMENT = 'GST treatment determined by the rules: TAXABLE, GST_FREE, INPUT_TAXED, REVERSE_CHARGE, EXPORT, NON_DEDUCTIBLE or OUT_OF_SCOPE.',
        gst.SOURCE_SYSTEM AS SOURCE_SYSTEM
            WITH SYNONYMS = ('source system','ERP','system','source')
            COMMENT = 'Originating ERP: SAP, ORACLE or LEGACY.',
        gst.RECON_STATUS AS RECON_STATUS
            WITH SYNONYMS = ('reconciliation status','recon status','BAS status','match status')
            COMMENT = 'Reconciliation outcome against the lodged BAS: MATCHED, REVIEW or INVESTIGATE.',
        gst.SUPPLIER_NAME AS SUPPLIER_NAME
            WITH SYNONYMS = ('supplier','vendor','creditor','payee')
            COMMENT = 'Supplier of the acquisition or customer of the supply.',
        gst.SUPPLIER_ABN AS SUPPLIER_ABN
            WITH SYNONYMS = ('ABN','supplier ABN','Australian business number')
            COMMENT = 'Supplier ABN from the supplier master. NULL means no ABN is held, which denies the input tax credit above $82.50.',
        gst.EXCEPTION_CODES AS EXCEPTION_CODES
            WITH SYNONYMS = ('exception codes','issue codes','error codes','findings')
            COMMENT = 'Comma separated exception codes raised against the line, E1 to E7.',
        gst.MAX_SEVERITY AS MAX_SEVERITY
            WITH SYNONYMS = ('severity','risk level','priority')
            COMMENT = 'Highest severity among the exceptions on the line: HIGH, MEDIUM, LOW or NONE.',
        gst.IS_ACCOUNT_MAPPED AS IS_ACCOUNT_MAPPED
            WITH SYNONYMS = ('account mapped','mapped account','unmapped account','mapping complete')
            COMMENT = 'FALSE where the native account code has no entry in the account mapping table.',
        gst.IS_CROSS_BORDER AS IS_CROSS_BORDER
            WITH SYNONYMS = ('cross border','imported','offshore','overseas supplier')
            COMMENT = 'TRUE where the supplier country differs from the entity jurisdiction.'
    )
    METRICS (
        gst.LINE_COUNT AS COUNT(gst.GL_LINE_ID)
            WITH SYNONYMS = ('line count','number of lines','transactions','volume','how many')
            COMMENT = 'Number of ledger lines.',
        gst.TOTAL_NET_AMOUNT AS SUM(gst.NET_AMOUNT)
            WITH SYNONYMS = ('net amount','net','value excluding GST','ex GST amount','turnover')
            COMMENT = 'Total amount net of GST, in AUD.',
        gst.TOTAL_BOOKED_TAX AS SUM(gst.BOOKED_TAX_AMOUNT)
            WITH SYNONYMS = ('booked GST','posted GST','GST as booked','recorded GST')
            COMMENT = 'GST actually posted by the source ERP, in AUD.',
        gst.TOTAL_EXPECTED_TAX AS SUM(gst.EXPECTED_TAX_AMOUNT)
            WITH SYNONYMS = ('expected GST','computed GST','GST per the rules','correct GST','GST payable')
            COMMENT = 'GST expected under the determination rules, in AUD. This is the computed position.',
        gst.TOTAL_TAX_VARIANCE AS SUM(gst.TAX_VARIANCE)
            WITH SYNONYMS = ('variance','GST variance','difference','exposure','under or over')
            COMMENT = 'Booked GST minus expected GST, in AUD. Positive means over-booked.',
        gst.TOTAL_EXCEPTION_COUNT AS SUM(gst.EXCEPTION_COUNT)
            WITH SYNONYMS = ('exceptions','number of exceptions','issues','findings count','errors')
            COMMENT = 'Total exceptions raised across the selected lines.',
        gst.LODGED_NET_GST_AMOUNT AS MAX(gst.LODGED_NET_GST)
            WITH SYNONYMS = ('lodged GST','lodged net GST','BAS lodged','as lodged','net GST on the BAS')
            COMMENT = 'Net GST per the lodged BAS for the entity and period. Uses MAX because this figure repeats across every line in the period and must never be summed.',
        gst.COMPUTED_NET_GST_AMOUNT AS MAX(gst.COMPUTED_NET_GST)
            WITH SYNONYMS = ('computed net GST','calculated net GST','our GST position')
            COMMENT = 'Net GST computed from the ledger for the entity and period. Uses MAX because this figure repeats across every line in the period and must never be summed.'
    )
    COMMENT = 'Australian GST compliance semantic model over the governed ledger. Monthly periods, AUD, 14 Australian entities.';

### Create the agent

1. Open **AI & ML**, then **Agents**, then **Create agent**.
2. Name it `GST_COMPLIANCE_AGENT`.
3. Add a **Cortex Analyst** tool and point it at `AGENTS.SV_GST_COMPLIANCE`.
4. Set the response instructions to: *answer the question first, then show the SQL you generated, then state any assumptions you made about the tax period or the entity.*

That third instruction matters more than it looks. Half of the questions a tax specialist asks are implicitly scoped — "last quarter", "this month", "for the group" — and the difference between a useful agent and a dangerous one is whether it tells you which entity and which period it decided you meant. Showing the SQL is the other half: a tax reviewer who can read the query can sign off on the answer, and nobody signs off on a number with no working.

### Eight questions to ask it

1. What was our total GST payable last quarter?
2. Which entities have exceptions this month?
3. Show me net GST by entity for the last six months.
4. Which suppliers are missing an ABN on claimed credits?
5. What is our total exposure from unmapped accounts?
6. Compare computed GST to lodged GST by period.
7. Which supply categories generate the most exceptions?
8. **Show imported professional fees before and after 1 July 2024.** — demonstrate this one live.

> ✅ **Why question 8 is the one to run in front of the room**
>
> It exercises the effective-dated determination rules you built in Lab 1. The same population of imported professional fees, from the same offshore suppliers, on the same accounts, returns `GST_FREE` at 0.00% before 1 July 2024 and `REVERSE_CHARGE` at 10.00% from that date onward — because rule 9 stopped applying and rule 16 started.

> 💡 **The teaching point — and the point of the whole workshop**
>
> The agent is only as good as the semantic layer, and the semantic layer is only as good as the governed data underneath it. Point Cortex Analyst at three unreconciled ERPs with three date formats, three amount conventions and no entity crosswalk and it will answer fluently and wrongly.

---

## Step 6 — Materialise AI output for cost control

*~8 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-6)

Step 4 was the most expensive operation in this workshop. Every time somebody opened `AI_INSIGHTS.V_AI_EXCEPTION_RATIONALE`, the language model ran again — the same 25 exceptions, the same prompts, the same answers, billed afresh. A view recomputes on every read, and when the thing being recomputed is a model call, every read has a price and a wait attached to it.

This step writes the model output down once and lets the tax team read it as many times as they like.

- Declare a Dynamic Table with a 24-hour freshness objective over AI output.
- Understand why this particular Dynamic Table must read from base tables only.
- Confirm that reading it invokes no model at all.

> ⚠️ **Read the source list before you run this**
>
> This statement reads `RAW_DATA.DOC_TAX_INVOICE` and `RAW_DATA.DIM_SUPPLIER` and nothing else. That is deliberate, not an oversight.

### Block 1 — write the model output down once

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE AI_INSIGHTS.DT_AI_EXCEPTION_REVIEW
    TARGET_LAG = '24 hours'
    WAREHOUSE  = GSI_TAX_WH
AS
WITH doc_sample AS (
    -- Bound the population BEFORE the model sees it. 25 documents, joined to
    -- the supplier master so the prompt has something to reason about.
    SELECT d.DOC_ID,
           d.SOURCE_SYSTEM,
           d.SOURCE_DOC_REF,
           d.SUPPLIER_ID,
           d.BOOKED_SUPPLY_CATEGORY,
           d.INVOICE_NARRATIVE,
           s.SUPPLIER_NAME,
           s.SUPPLIER_ABN,
           s.GST_REGISTERED       AS SUPPLIER_GST_REGISTERED,
           s.SUPPLIER_COUNTRY
    FROM RAW_DATA.DOC_TAX_INVOICE d
    LEFT JOIN RAW_DATA.DIM_SUPPLIER s
           ON s.SUPPLIER_ID = d.SUPPLIER_ID
    LIMIT 25
),
classified AS (
    -- Same AI_CLASSIFY call and same six categories as Step 2.
    SELECT c.*,
           AI_CLASSIFY(
               c.INVOICE_NARRATIVE,
               ['taxable supply','GST-free supply','input taxed supply',
                'export','entertainment','capital acquisition']
           ):labels[0]::VARCHAR AS AI_SUPPLY_CLASSIFICATION
    FROM doc_sample c
),
flagged AS (
    SELECT c.*,
           -- Booked category translated into the vocabulary the model returns,
           -- so the two are comparable.
           CASE c.BOOKED_SUPPLY_CATEGORY
                WHEN 'ENTERTAINMENT'      THEN 'entertainment'
                WHEN 'CAPITAL_EQUIPMENT'  THEN 'capital acquisition'
                WHEN 'EXPORT_SALES'       THEN 'export'
                WHEN 'RESIDENTIAL_RENT'   THEN 'input taxed supply'
                WHEN 'FINANCIAL_SUPPLY'   THEN 'input taxed supply'
                WHEN 'FOOD_BASIC'         THEN 'GST-free supply'
                WHEN 'MEDICAL'            THEN 'GST-free supply'
                WHEN 'EDUCATION'          THEN 'GST-free supply'
                ELSE 'taxable supply'
           END AS BOOKED_AS_CLASSIFICATION,
           (c.AI_SUPPLY_CLASSIFICATION <>
            CASE c.BOOKED_SUPPLY_CATEGORY
                 WHEN 'ENTERTAINMENT'      THEN 'entertainment'
                 WHEN 'CAPITAL_EQUIPMENT'  THEN 'capital acquisition'
                 WHEN 'EXPORT_SALES'       THEN 'export'
                 WHEN 'RESIDENTIAL_RENT'   THEN 'input taxed supply'
                 WHEN 'FINANCIAL_SUPPLY'   THEN 'input taxed supply'
                 WHEN 'FOOD_BASIC'         THEN 'GST-free supply'
                 WHEN 'MEDICAL'            THEN 'GST-free supply'
                 WHEN 'EDUCATION'          THEN 'GST-free supply'
                 ELSE 'taxable supply'
            END) AS MISCODING_FLAG
    FROM classified c
)
SELECT
    f.DOC_ID,
    f.SOURCE_SYSTEM,
    f.SOURCE_DOC_REF,
    f.SUPPLIER_ID,
    f.SUPPLIER_NAME,
    f.SUPPLIER_COUNTRY,
    f.SUPPLIER_GST_REGISTERED,
    f.BOOKED_SUPPLY_CATEGORY,
    f.AI_SUPPLY_CLASSIFICATION,
    f.MISCODING_FLAG,
    -- Every concatenated column wrapped in COALESCE. One NULL nulls the whole
    -- prompt and the call returns NULL without complaining.
    AI_COMPLETE(
        'claude-sonnet-4-6',
        'You are an Australian indirect tax analyst preparing a review note. '  ||
        'In no more than three sentences, state whether the booked supply '     ||
        'category looks correct and what the GST consequence is if it is not. ' ||
        'Invoice narrative: '   || COALESCE(f.INVOICE_NARRATIVE, 'not supplied')      ||
        '. Booked category: '   || COALESCE(f.BOOKED_SUPPLY_CATEGORY, 'unmapped')     ||
        '. Category suggested by classification: '
                                || COALESCE(f.AI_SUPPLY_CLASSIFICATION, 'undetermined') ||
        '. Supplier: '          || COALESCE(f.SUPPLIER_NAME, 'unknown')              ||
        ' in '                  || COALESCE(f.SUPPLIER_COUNTRY, 'an unknown country') ||
        '. Supplier ABN: '      || COALESCE(f.SUPPLIER_ABN, 'not held')              ||
        '.'
    ) AS AI_REVIEW_NOTE
FROM flagged f;

The `LIMIT 25` sits inside `doc_sample`, above the model call, so it bounds the rows the model is asked to read rather than the rows returned to the screen. A `LIMIT` in the outer `SELECT` would have run the model 300 times and then thrown 275 answers away.

> 💡 **What `TARGET_LAG = '24 hours'` is actually promising**
>
> It is a statement about staleness, not a schedule. Snowflake undertakes to keep this table no more than 24 hours behind its sources, and works out for itself when to do the work.

### Block 2 — read it, and confirm no model ran

In [ ]:
-- 1. The review queue, highest-interest documents first. This is now an
--    ordinary table read: 25 rows off storage, no model invocation, no
--    per-token cost, and it returns in the time any small SELECT takes.
SELECT DOC_ID,
       SUPPLIER_NAME,
       BOOKED_SUPPLY_CATEGORY,
       AI_SUPPLY_CLASSIFICATION,
       MISCODING_FLAG,
       AI_REVIEW_NOTE
FROM AI_INSIGHTS.DT_AI_EXCEPTION_REVIEW
ORDER BY MISCODING_FLAG DESC, DOC_ID
LIMIT 10;

In [ ]:
-- 2. What Snowflake decided to do with your declaration.
SHOW DYNAMIC TABLES IN SCHEMA AI_INSIGHTS;

Run statement 1 twice and watch the elapsed time. Then compare it against Step 4 Block 3, which asked the same question of the view: there, each execution queued 25 prompts, waited for 25 completions and paid for every token in both directions. Here the text was generated once and is now being read, which is why the second execution is quicker still — Snowflake serves it from the result cache and consumes no compute at all.

| Reading the same 25 working papers | Model calls per read | What the tax team experiences |
|---|---|---|
| From the view built in Step 4 | 25, every single time | A wait of seconds, and a bill, on each refresh of the screen. Ten reviewers reading it once each is 250 calls. |
| From `DT_AI_EXCEPTION_REVIEW` | **0** | Immediate. Ten reviewers reading it ten times each is still zero calls until the table next refreshes. |

Read the `refresh_mode` column in the output of statement 2 rather than assuming a value. `AI_COMPLETE` is not a deterministic function, so Snowflake cannot track changes through it incrementally and will report a full refresh. That is the correct answer for this object and it costs nothing here, because a full refresh of 25 documents is 25 calls once a day. It is also the reason the bounding `LIMIT` matters: whatever the population is, the refresh will process all of it.

> ✅ **The production shape**
>
> Classify and draft on arrival, write the answer down, and let everything downstream read the stored text. The generated narrative in this table carries `DOC_ID`, `SOURCE_SYSTEM` and `SOURCE_DOC_REF`, so it still joins back to the ledger and to the exception register — the audit trail survives being written down.

---

## Step 7 — Triage taxpayer correspondence

*~8 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-7)

Every exception you have raised so far generates correspondence, and correspondence arrives faster than anyone can triage it. Some of it is a supplier confirming a corrected invoice; some of it is a business unit disputing a finding three days before lodgement. Both look identical in an inbox.

`CORRESPONDENCE_TEXT` on `RAW_DATA.DOC_TAX_INVOICE` holds the message attached to each document. This step reads the tone of it, so the queue can be worked in the order the tax team should actually work it.

- Return a sentiment label from free text with one function call.
- Handle all four labels the function can return.
- Turn the label into a triage priority the tax team can sort on.

> ⚠️ **`AI_SENTIMENT` returns `mixed` as well as positive, negative and neutral**
>
> This is the single most commonly missed detail with this function. A `CASE` that handles only `positive`, `negative` and `neutral` does not fail, does not warn and does not return NULL — it silently drops every `mixed` row into whatever your `ELSE` branch says, and `mixed` is exactly the label you get on the messages that matter most: a supplier …

### Block 1 — read the tone, next to the text

In [ ]:
WITH correspondence AS (
    -- Bound the population before the model sees it, as in every other step.
    SELECT DOC_ID,
           SOURCE_SYSTEM,
           SOURCE_DOC_REF,
           SUPPLIER_ID,
           BOOKED_SUPPLY_CATEGORY,
           CORRESPONDENCE_TEXT
    FROM RAW_DATA.DOC_TAX_INVOICE
    WHERE CORRESPONDENCE_TEXT IS NOT NULL
    LIMIT 25
)
SELECT
    DOC_ID,
    CORRESPONDENCE_TEXT,
    AI_SENTIMENT(CORRESPONDENCE_TEXT):categories[0]:sentiment::VARCHAR AS SENTIMENT_LABEL,
    -- All four labels. Not three.
    CASE AI_SENTIMENT(CORRESPONDENCE_TEXT):categories[0]:sentiment::VARCHAR
         WHEN 'negative' THEN '1 - URGENT'
         WHEN 'mixed'    THEN '2 - REVIEW'
         WHEN 'neutral'  THEN '3 - ROUTINE'
         WHEN 'positive' THEN '4 - MONITOR'
         ELSE                 '9 - UNCLASSIFIED'
    END AS TRIAGE_PRIORITY
FROM correspondence
ORDER BY TRIAGE_PRIORITY, DOC_ID;

Read the second and third columns together for a minute. The negative messages are the ones disputing a finding or escalating a delay, and they are the ones with a deadline attached. The `mixed` rows are the interesting ones: a message that thanks the team for the correction and then objects to the amount is genuinely both, and forcing it into `positive` or `negative` would lose the half that needs a response.

| Label | Triage priority | What it usually means in practice |
|---|---|---|
| `negative` | **1 - URGENT** | A dispute, an escalation or a complaint. Someone is waiting and unhappy about it. |
| `mixed` | **2 - REVIEW** | Co-operative on one point and objecting on another. Needs a human to read it properly — and the branch most people forget to write. |
| `neutral` | **3 - ROUTINE** | A factual note. Acknowledge and file. |
| `positive` | **4 - MONITOR** | A confirmation that something has been resolved. No action beyond closing the item. |

### Block 2 — persist it, and see the shape of the queue

In [ ]:
CREATE OR REPLACE VIEW AI_INSIGHTS.V_AI_CORRESPONDENCE_TRIAGE AS
WITH correspondence AS (
    SELECT DOC_ID,
           SOURCE_SYSTEM,
           SOURCE_DOC_REF,
           SUPPLIER_ID,
           BOOKED_SUPPLY_CATEGORY,
           CORRESPONDENCE_TEXT
    FROM RAW_DATA.DOC_TAX_INVOICE
    WHERE CORRESPONDENCE_TEXT IS NOT NULL
    LIMIT 25
),
scored AS (
    SELECT c.*,
           AI_SENTIMENT(c.CORRESPONDENCE_TEXT):categories[0]:sentiment::VARCHAR AS SENTIMENT_LABEL
    FROM correspondence c
)
SELECT
    s.DOC_ID,
    s.SOURCE_SYSTEM,
    s.SOURCE_DOC_REF,
    s.SUPPLIER_ID,
    s.BOOKED_SUPPLY_CATEGORY,
    s.CORRESPONDENCE_TEXT,
    s.SENTIMENT_LABEL,
    CASE s.SENTIMENT_LABEL
         WHEN 'negative' THEN '1 - URGENT'
         WHEN 'mixed'    THEN '2 - REVIEW'
         WHEN 'neutral'  THEN '3 - ROUTINE'
         WHEN 'positive' THEN '4 - MONITOR'
         ELSE                 '9 - UNCLASSIFIED'
    END AS TRIAGE_PRIORITY,
    (s.SENTIMENT_LABEL IN ('negative', 'mixed')) AS NEEDS_RESPONSE
FROM scored s;

In [ ]:
-- The queue by priority. If '9 - UNCLASSIFIED' returns any rows, the CASE is
-- missing a label the function actually emitted - go and look at it.
SELECT TRIAGE_PRIORITY,
       SENTIMENT_LABEL,
       COUNT(*)                                  AS MESSAGE_COUNT,
       COUNT(DISTINCT SUPPLIER_ID)               AS SUPPLIERS,
       SUM(IFF(NEEDS_RESPONSE, 1, 0))            AS AWAITING_RESPONSE
FROM AI_INSIGHTS.V_AI_CORRESPONDENCE_TRIAGE
GROUP BY 1, 2
ORDER BY 1;

The `'9 - UNCLASSIFIED'` branch is a deliberate canary rather than defensive padding. Because all four documented labels are handled above it, any row landing there means the function returned something the view does not know about — and you will see it as a count rather than discovering it months later as a silently mis-prioritised message.

> 💡 **Sentiment is a routing signal, not a tax position**
>
> Nothing in this step changes a treatment, a rate or a return. An angry supplier is not evidence that the GST is wrong, and a grateful one is not evidence that it is right — the determination rules from Lab 1 remain the only thing that decides the tax outcome.

> ✅ **Where you are**
>
> Three kinds of unstructured evidence are now queryable: the narrative told you what was bought, the remarks told you what the invoice claimed, and the correspondence tells you how hard someone is pushing back. All three join to the ledger on `(SOURCE_SYSTEM, SOURCE_DOC_REF)`, which is what keeps them part of the tax record rather than a separate pile.

---

## Step 8 — Add an AI review queue to the app

*~10 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-8)

The control tower you built in Lab 1 shows the tax team what is wrong. It does not help them work through it, because the one thing a reviewer needs in front of them — why this line was flagged, in words, and which rule decided it — was not available until Step 4 of this lab.

This is the same app with a fourth tab. Replace the whole file: open your Streamlit app in Snowsight (**Projects → Streamlit**), select all of the existing code, and paste this over it.

- Put the generated working paper in front of the reviewer, beside the rule that fired.
- Capture an approve or reject decision per exception.
- Keep every query an aggregate or a bounded read, against a 2.4 million line ledger.

In [ ]:
# Streamlit app code. Run this cell to preview the app inline, or paste it
# into Projects -> Streamlit to deploy it standalone (add st.set_page_config
# back at the top when you do).
import pandas as pd
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Streamlit in Snowflake hands you the session. Never build a connection.
session = get_active_session()

DB = "GSI_TAX_WORKSHOP"


# ----------------------------------------------------------------------------
# Cached queries. Every one of these aggregates or bounds its result set - none
# of them scans 2.4 million rows back to the browser.
# ----------------------------------------------------------------------------
@st.cache_data(ttl=600)
def load_group_totals():
    return session.sql(f"""
        SELECT SUM(COMPUTED_NET_GST)                        AS COMPUTED_NET_GST,
               SUM(LODGED_NET_GST)                          AS LODGED_NET_GST,
               SUM(VARIANCE_NET)                            AS VARIANCE_NET,
               COUNT(*)                                     AS ENTITY_PERIODS,
               SUM(IFF(RECON_STATUS = 'MATCHED', 1, 0))     AS MATCHED,
               SUM(IFF(RECON_STATUS = 'INVESTIGATE', 1, 0)) AS INVESTIGATE
        FROM {DB}.TAX.V_GST_VS_BAS
    """).to_pandas()


@st.cache_data(ttl=600)
def load_reconciliation():
    return session.sql(f"""
        SELECT ENTITY_NAME, TAX_PERIOD, COMPUTED_NET_GST, LODGED_NET_GST,
               VARIANCE_NET, VARIANCE_PCT, RECON_STATUS
        FROM {DB}.TAX.V_GST_VS_BAS
        ORDER BY ABS(VARIANCE_NET) DESC
    """).to_pandas()


@st.cache_data(ttl=600)
def load_exception_summary():
    return session.sql(f"""
        SELECT MAX_SEVERITY,
               COUNT(*)              AS LINE_COUNT,
               SUM(ABS(TAX_VARIANCE)) AS ABS_VARIANCE
        FROM {DB}.TAX.V_GST_EXCEPTIONS
        GROUP BY 1
    """).to_pandas()


@st.cache_data(ttl=600)
def load_exceptions(severity, limit):
    where = "" if severity == "ALL" else f"WHERE MAX_SEVERITY = '{severity}'"
    return session.sql(f"""
        SELECT GL_LINE_ID, ENTITY_NAME, TAX_PERIOD, SUPPLIER_NAME,
               SUPPLY_CATEGORY, TAX_TREATMENT, APPLIED_RULE_ID,
               NET_AMOUNT, BOOKED_TAX_AMOUNT, EXPECTED_TAX_AMOUNT,
               TAX_VARIANCE, EXCEPTION_CODES, MAX_SEVERITY, RISK_SCORE
        FROM {DB}.TAX.V_GST_EXCEPTIONS
        {where}
        ORDER BY RISK_SCORE DESC
        LIMIT {int(limit)}
    """).to_pandas()


@st.cache_data(ttl=600)
def load_heatmap():
    return session.sql(f"""
        SELECT ENTITY_NAME, TAX_PERIOD, COUNT(*) AS EXCEPTION_LINES
        FROM {DB}.TAX.V_GST_EXCEPTIONS
        GROUP BY 1, 2
    """).to_pandas()


@st.cache_data(ttl=600)
def load_ai_queue():
    # 25 rows, generated in Step 4. If you named the generated column something
    # else in your own view, change AI_WORKING_PAPER to match it.
    return session.sql(f"""
        SELECT GL_LINE_ID, ENTITY_NAME, SUPPLIER_NAME, EXCEPTION_CODES,
               MAX_SEVERITY, TAX_VARIANCE, APPLIED_RULE_ID,
               APPLIED_RULE_DESCRIPTION, AI_WORKING_PAPER
        FROM {DB}.AI_INSIGHTS.V_AI_EXCEPTION_RATIONALE
        ORDER BY MAX_SEVERITY, ABS(TAX_VARIANCE) DESC
    """).to_pandas()


def money(value):
    return f"${value:,.0f}"


# ----------------------------------------------------------------------------
# Header tiles
# ----------------------------------------------------------------------------
st.title("GST Control Tower")
st.caption("One governed ledger, the group GST position, and the review queue.")

totals = load_group_totals().iloc[0]
severity = load_exception_summary()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Computed net GST", money(totals["COMPUTED_NET_GST"]))
c2.metric("Lodged net GST", money(totals["LODGED_NET_GST"]))
c3.metric("Net variance", money(totals["VARIANCE_NET"]),
          delta=f"{int(totals['INVESTIGATE'])} to investigate", delta_color="inverse")
c4.metric("Exception lines", f"{int(severity['LINE_COUNT'].sum()):,}")

tab_bas, tab_exc, tab_heat, tab_ai = st.tabs(
    ["BAS reconciliation", "Exception register", "Entity heatmap", "AI review queue"]
)


# ----------------------------------------------------------------------------
# Tab 1 - BAS reconciliation
# ----------------------------------------------------------------------------
with tab_bas:
    st.subheader("Computed against lodged, by entity and period")
    recon = load_reconciliation()

    def rag(row):
        colour = {"MATCHED": "#E8F7EE",
                  "REVIEW": "#FFF6E0",
                  "INVESTIGATE": "#FDE8E8"}.get(row["RECON_STATUS"], "")
        return [f"background-color: {colour}"] * len(row)

    st.dataframe(
        recon.style.apply(rag, axis=1).format(
            {"COMPUTED_NET_GST": "{:,.2f}", "LODGED_NET_GST": "{:,.2f}",
             "VARIANCE_NET": "{:,.2f}", "VARIANCE_PCT": "{:,.2f}"}
        ),
        use_container_width=True, height=460
    )
    st.caption(
        f"{int(totals['MATCHED'])} of {int(totals['ENTITY_PERIODS'])} entity-periods "
        "reconcile to the lodged BAS. The rest are ranked by absolute variance."
    )


# ----------------------------------------------------------------------------
# Tab 2 - Exception register
# ----------------------------------------------------------------------------
with tab_exc:
    st.subheader("Exceptions, highest risk score first")
    f1, f2 = st.columns([1, 1])
    pick = f1.selectbox("Severity", ["ALL", "HIGH", "MEDIUM", "LOW"])
    rows = f2.slider("Rows to show", 50, 500, 200, step=50)

    st.dataframe(load_exceptions(pick, rows), use_container_width=True, height=460)
    st.dataframe(severity, use_container_width=True)
    st.caption(
        "The register is always read with ORDER BY RISK_SCORE DESC and a LIMIT. "
        "Nobody works a queue of 253,394 lines from the top."
    )


# ----------------------------------------------------------------------------
# Tab 3 - Entity heatmap
# ----------------------------------------------------------------------------
with tab_heat:
    st.subheader("Exception lines by entity and tax period")
    heat = load_heatmap()
    grid = heat.pivot(index="ENTITY_NAME", columns="TAX_PERIOD",
                      values="EXCEPTION_LINES").fillna(0).astype(int)
    st.dataframe(
        grid.style.background_gradient(cmap="Blues", axis=None),
        use_container_width=True, height=460
    )
    st.caption("Read it across for a period that went wrong everywhere, "
               "and down for an entity with a standing problem.")


# ----------------------------------------------------------------------------
# Tab 4 - AI review queue
# ----------------------------------------------------------------------------
with tab_ai:
    st.subheader("Working papers awaiting review")
    queue = load_ai_queue()

    if queue.empty:
        st.warning("No rows in AI_INSIGHTS.V_AI_EXCEPTION_RATIONALE. "
                   "Run Step 4 of this lab first.")
    else:
        if "decisions" not in st.session_state:
            st.session_state["decisions"] = {}

        labels = [f"{r.GL_LINE_ID} - {r.MAX_SEVERITY} - {r.EXCEPTION_CODES}"
                  for r in queue.itertuples()]
        chosen = st.selectbox(f"Exception ({len(queue)} in the queue)", labels)
        row = queue.iloc[labels.index(chosen)]

        left, right = st.columns([2, 1])

        with left:
            st.markdown("**Generated working paper**")
            st.info(row["AI_WORKING_PAPER"])
            st.markdown("**Rule applied**")
            st.write(f"Rule {int(row['APPLIED_RULE_ID'])} - "
                     f"{row['APPLIED_RULE_DESCRIPTION']}")

        with right:
            st.metric("Tax variance", money(row["TAX_VARIANCE"]))
            st.write(f"**Entity** {row['ENTITY_NAME']}")
            st.write(f"**Supplier** {row['SUPPLIER_NAME']}")

            current = st.session_state["decisions"].get(row["GL_LINE_ID"], "Not reviewed")
            decision = st.radio(
                "Reviewer decision",
                ["Not reviewed", "Approve - treatment is correct",
                 "Reject - refer for adjustment"],
                index=["Not reviewed", "Approve - treatment is correct",
                       "Reject - refer for adjustment"].index(current),
                key=f"decision_{row['GL_LINE_ID']}"
            )
            st.session_state["decisions"][row["GL_LINE_ID"]] = decision

        reviewed = {k: v for k, v in st.session_state["decisions"].items()
                    if v != "Not reviewed"}
        st.divider()
        st.caption(f"{len(reviewed)} of {len(queue)} reviewed in this session.")
        if reviewed:
            st.dataframe(
                pd.DataFrame({"GL_LINE_ID": list(reviewed.keys()),
                              "DECISION": list(reviewed.values())}),
                use_container_width=True
            )

### What the fourth tab changes about the reviewer's job

The first three tabs answer *what is wrong*. The fourth answers *why*, and it does so in the order that matters: the generated working paper first, the rule that produced the treatment immediately beneath it, and the variance and counterparty to the side. A reviewer can form a view without leaving the screen or writing a query, and the rule identifier means the sentence they are reading is traceable to a row in `TAX.TAX_RULE_DETERMINATION` that a person can inspect and, if it is wrong, change.

> 💡 **The decisions are in session state, deliberately**
>
> `st.session_state` holds the approve and reject choices for the duration of the session and no longer. That is the honest shape for a workshop: persisting a reviewer's sign-off means an audit table, a user identity, a timestamp and a policy on who may overturn whom, which is a governance conversation rather than a Streamlit one.

> ⚠️ **Why every query here is an aggregate or a bounded read**
>
> Nothing in this app returns a row per ledger line. The tiles and the heatmap are `GROUP BY` aggregates, the reconciliation reads a 532-row view, the register carries an explicit `LIMIT` that the user controls, and the AI queue is the 25 rows Step 4 bounded.

> ✅ **Where you are**
>
> One application, four tabs, over a governed ledger: the group position, the reconciliation against what was lodged, the exception register ranked by risk, and a review queue where the explanation is already written. The tax team never sees the SQL, and every number on the screen traces back to a rule row and a source document reference.

---

## Step 9 — A second tool on the agent

*~8 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-9)

Your agent answers questions about the group GST position because it has one tool: an Analyst tool over `AGENTS.SV_GST_COMPLIANCE`. Ask it something that semantic view cannot express — *which suppliers are the riskiest to keep claiming credits from?* — and it will do its best with the dimensions it has, which is not the same as answering the question.

Give it a second tool and it stops guessing. But a second tool only helps if the agent can tell which of the two to reach for.

- Build a supplier-risk view scoped to the question a second tool should answer.
- Attach it as a second Analyst tool on the existing agent.
- Write tool descriptions that make the routing decision unambiguous.

### Block 1 — the supplier-risk view

In [ ]:
CREATE OR REPLACE VIEW AGENTS.V_SUPPLIER_RISK AS
WITH ledger AS (
    -- One row per supplier: what we buy from them and what it is worth.
    SELECT SUPPLIER_ID,
           SUPPLIER_NAME,
           SUPPLIER_COUNTRY,
           SUPPLIER_GST_REGISTERED,
           SUPPLIER_ABN,
           COUNT(*)                                              AS LINE_COUNT,
           COUNT(DISTINCT ENTITY_ID)                             AS ENTITIES_TRANSACTING,
           MIN(TAX_PERIOD)                                       AS FIRST_PERIOD,
           MAX(TAX_PERIOD)                                       AS LAST_PERIOD,
           SUM(NET_AMOUNT)                                       AS NET_AMOUNT,
           SUM(BOOKED_TAX_AMOUNT)                                AS BOOKED_TAX_AMOUNT,
           SUM(EXPECTED_TAX_AMOUNT)                              AS EXPECTED_TAX_AMOUNT,
           SUM(IFF(RECOVERABLE AND DOC_TYPE = 'AP_INVOICE',
                   EXPECTED_TAX_AMOUNT, 0))                      AS CREDITS_CLAIMED,
           SUM(IFF(IS_CROSS_BORDER, 1, 0))                       AS CROSS_BORDER_LINES
    FROM TAX.V_GST_DETERMINED
    GROUP BY ALL
),
exceptions AS (
    SELECT SUPPLIER_ID,
           COUNT(*)                                              AS EXCEPTION_LINES,
           SUM(IFF(MAX_SEVERITY = 'HIGH', 1, 0))                 AS HIGH_SEVERITY_LINES,
           SUM(ABS(COALESCE(TAX_VARIANCE, 0)))                   AS ABS_TAX_VARIANCE,
           LISTAGG(DISTINCT EXCEPTION_CODES, ', ')               AS EXCEPTION_CODE_MIX
    FROM TAX.V_GST_EXCEPTIONS
    GROUP BY SUPPLIER_ID
)
SELECT
    l.SUPPLIER_ID,
    l.SUPPLIER_NAME,
    l.SUPPLIER_COUNTRY,
    l.SUPPLIER_GST_REGISTERED,
    (l.SUPPLIER_ABN IS NULL)                                     AS ABN_MISSING,
    l.ENTITIES_TRANSACTING,
    l.FIRST_PERIOD,
    l.LAST_PERIOD,
    l.LINE_COUNT,
    l.CROSS_BORDER_LINES,
    l.NET_AMOUNT,
    l.BOOKED_TAX_AMOUNT,
    l.EXPECTED_TAX_AMOUNT,
    l.CREDITS_CLAIMED,
    COALESCE(e.EXCEPTION_LINES, 0)                               AS EXCEPTION_LINES,
    COALESCE(e.HIGH_SEVERITY_LINES, 0)                           AS HIGH_SEVERITY_LINES,
    COALESCE(e.ABS_TAX_VARIANCE, 0)                              AS ABS_TAX_VARIANCE,
    e.EXCEPTION_CODE_MIX,
    ROUND(100.0 * COALESCE(e.EXCEPTION_LINES, 0) / l.LINE_COUNT, 2)
                                                                 AS PCT_LINES_EXCEPTED,
    -- Credit exposure: what we have claimed from a supplier we cannot
    -- substantiate a claim against, because there is no ABN on file.
    IFF(l.SUPPLIER_ABN IS NULL, l.CREDITS_CLAIMED, 0)            AS UNSUBSTANTIATED_CREDITS,
    CASE WHEN l.SUPPLIER_ABN IS NULL AND l.CREDITS_CLAIMED > 0        THEN 'HIGH'
         WHEN NOT l.SUPPLIER_GST_REGISTERED AND l.BOOKED_TAX_AMOUNT <> 0 THEN 'HIGH'
         WHEN COALESCE(e.HIGH_SEVERITY_LINES, 0) > 0                  THEN 'MEDIUM'
         WHEN COALESCE(e.EXCEPTION_LINES, 0) > 0                      THEN 'LOW'
         ELSE 'NONE'
    END                                                          AS SUPPLIER_RISK_BAND
FROM ledger l
LEFT JOIN exceptions e ON e.SUPPLIER_ID = l.SUPPLIER_ID;

Two thousand rows, one per supplier, and every column answers a question a tax person would actually ask about a counterparty: how much we buy, how much we have claimed, whether we hold the ABN that entitles us to claim it, and how often lines from this supplier end up in the register. `UNSUBSTANTIATED_CREDITS` is the number that gets attention — input tax credits claimed where no ABN is held anywhere in the group.

### Attach it as a second Analyst tool

Build a semantic view over `AGENTS.V_SUPPLIER_RISK` the same way you built the first one (**AI & ML → Cortex Analyst → Create new**), name it `SV_SUPPLIER_RISK`, and take the Autopilot suggestions: supplier, country, registration status and risk band as dimensions; line count, net amount, credits claimed, unsubstantiated credits and exception lines as metrics. Then open **AI & ML → Agents**, edit `GST_COMPLIANCE_AGENT` and add a second Analyst tool pointing at it.

Add the synonyms while you are in the semantic view. A tax person types "vendor", not `SUPPLIER_NAME`, and asks about "unsubstantiated credits" or "credits at risk", not `UNSUBSTANTIATED_CREDITS`. The synonyms are what make the tool usable by the person who has the question.

### How the agent decides which tool to use

There is no routing table and nothing to configure. The agent reads the description of each tool, compares it against what the user asked, and picks. That makes the description the whole mechanism — it is the only thing standing between a well-formed question and an answer drawn from the wrong data.

| Tool | Description to give it | Questions it should win |
|---|---|---|
| `SV_GST_COMPLIANCE` | The group GST position by legal entity, tax period, supply category and tax treatment, including computed GST, GST lodged on the BAS, the variance between them and exception counts. Use for questions about periods, returns, reconciliation and the group position. | "Net GST by entity for the last six months." "Compare computed GST to what we lodged." |
| `SV_SUPPLIER_RISK` | One row per supplier covering spend, input tax credits claimed, whether an ABN is held, GST registration status, cross-border lines and a supplier risk band. Use for questions about individual suppliers, vendors or counterparties and the credits claimed from them. | "Which suppliers are missing an ABN on claimed credits?" "Who are our riskiest vendors?" |

Try three questions in the agent and watch which tool it names in its reasoning. *What was our net GST last quarter?* should take the compliance tool. *Which suppliers have we claimed credits from without an ABN?* should take the supplier tool. Then ask *which entities are claiming credits from unregistered suppliers?* — that spans both, and the agent will either query one and reason from it or call both in sequence. Read what it says it did rather than assuming.

> ⚠️ **Vague descriptions are the failure mode, not vague questions**
>
> Describe both tools as "GST data" and the agent has nothing to choose on. It will pick one, answer confidently and give an answer that is wrong in a way nobody can see, because the numbers will be real numbers from a real view — just not the view that could answer the question.

> 💡 **Two tools, one governed source**
>
> Both semantic views resolve to `TAX.V_GST_DETERMINED` and therefore to the same rule tables and the same conformed ledger. That is what stops the second tool becoming a second version of the truth: a supplier's credits reconcile to the entity's return because they are the same lines, aggregated differently.

---

## Step 10 — Clean up

*~2 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab2_cortex_ai.html#step-10)

Everything built across both labs lives in two databases, one warehouse and — if you completed the sharing optional in Lab 1 — one share. This removes all of it.

> ⚠️ **This is irreversible beyond the Time Travel window**
>
> Dropping a database starts its Time Travel clock, and `UNDROP DATABASE` works only until the retention period expires. Past that point the objects, the 2.4 million ledger lines, the rule tables and every view over them are gone with no recovery path.

In [ ]:
-- Drop the share first. A share cannot be dropped implicitly with the
-- database it grants on, and leaving it behind leaves a dangling grant.
DROP SHARE IF EXISTS GSI_TAX_AUDIT_SHARE;

-- The scenario clone from Lab 1 Optional. Harmless if it was already dropped
-- at the end of that step, or never created.
DROP DATABASE IF EXISTS GSI_TAX_SCENARIO;

-- The workshop itself: all seven schemas, the Dynamic Table, every view,
-- the rule tables and the Streamlit app object.
DROP DATABASE IF EXISTS GSI_TAX_WORKSHOP;

-- The warehouse last, so the statements above have compute to run on.
DROP WAREHOUSE IF EXISTS GSI_TAX_WH;

In [ ]:
-- Confirm. Both queries should return zero rows.
SHOW DATABASES LIKE 'GSI_TAX%';

SHOW WAREHOUSES LIKE 'GSI_TAX%';

Every statement uses `IF EXISTS`, so the block runs clean whether or not you completed the optional steps. Dropping `GSI_TAX_WORKSHOP` takes the Streamlit app with it, because the app object lives in the `APPS` schema.

> 💡 **The agent and the semantic views need removing separately**
>
> `GST_COMPLIANCE_AGENT` was created through Snowsight and is not an object inside `GSI_TAX_WORKSHOP`, so dropping the database does not remove it. Open **AI & ML → Agents**, select the agent and delete it there.

> ✅ **What you take with you**
>
> The environment is disposable; the pattern is not. Three source systems conformed into one ledger that holds no tax logic, an effective-dated rule set the tax team owns, a determination layer thin enough to audit, exceptions ranked by risk, and language models supplying the attributes the rules were missing rather than replacing the rules.